# Block Size Analysis

This notebook analyzes blockchain block sizes from CSV data generated by the block size benchmarking scripts.

## Setup

First, generate data using the block size analysis script:
```bash
python3 ../block_size_benchmarks/analyze_block_sizes.py --url ws://127.0.0.1:9944 --latest-n 100
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Load Data

Update the path to your CSV file:

In [ ]:
# Update this path to your CSV file
csv_file = "../block_size_benchmarks/block_size_analysis_latest_100/block_sizes.csv"

# Load data
df = pd.read_csv(csv_file)

# Display basic info
print(f"Loaded {len(df)} blocks")
print(f"Block range: #{df['block_number'].min()} to #{df['block_number'].max()}")
df.head()

## Summary Statistics

In [ ]:
print("=" * 60)
print("BLOCK SIZE STATISTICS")
print("=" * 60)
print(f"Average block size:     {df['size_kb'].mean():.2f} KB")
print(f"Median block size:      {df['size_kb'].median():.2f} KB")
print(f"Min block size:         {df['size_kb'].min():.2f} KB")
print(f"Max block size:         {df['size_kb'].max():.2f} KB")
print(f"Std deviation:          {df['size_kb'].std():.2f} KB")
print(f"Total data:             {df['size_mb'].sum():.2f} MB")
print()
print("EXTRINSIC STATISTICS")
print("=" * 60)
print(f"Total extrinsics:       {df['extrinsic_count'].sum():,}")
print(f"Average per block:      {df['extrinsic_count'].mean():.1f}")
print(f"Min per block:          {df['extrinsic_count'].min()}")
print(f"Max per block:          {df['extrinsic_count'].max()}")

## Block Size Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df['block_number'], df['size_kb'], marker='o', markersize=4, linewidth=1.5, alpha=0.7)
ax.axhline(y=df['size_kb'].mean(), color='r', linestyle='--', linewidth=2, label=f"Average: {df['size_kb'].mean():.2f} KB")

ax.set_xlabel('Block Number', fontsize=12)
ax.set_ylabel('Block Size (KB)', fontsize=12)
ax.set_title('Block Size Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Block Size Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
ax1.hist(df['size_kb'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(df['size_kb'].mean(), color='r', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(df['size_kb'].median(), color='g', linestyle='--', linewidth=2, label='Median')
ax1.set_xlabel('Block Size (KB)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Block Size Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Box plot
ax2.boxplot(df['size_kb'], vert=True, patch_artist=True, 
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
ax2.set_ylabel('Block Size (KB)', fontsize=12)
ax2.set_title('Block Size Box Plot', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Block Size vs Extrinsic Count

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(df['extrinsic_count'], df['size_kb'], alpha=0.6, s=60, edgecolors='black', linewidth=0.5)

# Add trend line
if len(df) > 1:
    z = np.polyfit(df['extrinsic_count'], df['size_kb'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(df['extrinsic_count'].min(), df['extrinsic_count'].max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", linewidth=2, label=f'Trend: y = {z[0]:.2f}x + {z[1]:.2f}')

ax.set_xlabel('Number of Extrinsics', fontsize=12)
ax.set_ylabel('Block Size (KB)', fontsize=12)
ax.set_title('Block Size vs Number of Extrinsics', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Top 10 Largest Blocks

In [ ]:
top_blocks = df.nlargest(10, 'size_kb')[['block_number', 'size_kb', 'extrinsic_count']]
top_blocks.reset_index(drop=True, inplace=True)
top_blocks.index = top_blocks.index + 1
print("Top 10 Largest Blocks:")
print(top_blocks.to_string())

## Size per Extrinsic Analysis

In [ ]:
# Calculate average size per extrinsic for each block
df['size_per_extrinsic'] = df['size_kb'] / df['extrinsic_count'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df['block_number'], df['size_per_extrinsic'], marker='o', markersize=4, linewidth=1.5, alpha=0.7, color='orange')
ax.axhline(y=df['size_per_extrinsic'].mean(), color='r', linestyle='--', linewidth=2, 
           label=f"Average: {df['size_per_extrinsic'].mean():.2f} KB/extrinsic")

ax.set_xlabel('Block Number', fontsize=12)
ax.set_ylabel('KB per Extrinsic', fontsize=12)
ax.set_title('Average Size per Extrinsic Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Custom Analysis

Add your own analysis cells below:

In [ ]:
# Your custom analysis here